In [ ]:
# @author: lajello
# A collection of examples of how to use APIs and API wrappers.

## Setup: authentication and cost

**IMPORTANT DISCLAIMER: if you would like to use the Google Cloud API, you will do it at your own risk. You will be the only person responsible for any incurred costs. Using this API is NOT necessary to pass the exam with full grades.**

Using this API requires a Google Cloud account with billing enabled (a credit card is
needed for verification), and a project with the Speech-to-Text API turned on. See the
note below before you start.

### How to authenticate

You need an API key from a Google Cloud project that has billing enabled:

1. In the Google Cloud console (https://console.cloud.google.com) create a project and
   enable billing for it.
2. Enable the API for that project: **APIs & Services -> Enable APIs and services ->
   search for "speech" -> Enable** on *Cloud Speech-to-Text API*.
3. Create the key: **APIs & Services -> Credentials -> Create credentials -> API key**.
4. Restrict it: **Edit API key -> API restrictions -> Cloud Speech-to-Text API**, so the key
   is useless for anything else if it ever leaks.
5. Paste the key into the `google_api_key` variable in the setup cell below.

Treat the key like a password. It is attached to a project with billing enabled, so **never
commit it to git** and never share it with anyone.

### How often you can run this for free

Speech-to-Text has a permanent free tier of **60 minutes of audio per month** per account
(it does not expire with the 90 day trial). The audio file used here is 27 seconds, and the
notebook sends it **twice**, once to the main API and once to the beta API. Billing rounds
each call up to the nearest 15 seconds, so one full run of the notebook costs you about
**1 minute** of the free hour.

**In other words: you can run this notebook about 60 times per month for free.** A few runs
is all you need to solve the exercise, so do not put these API calls inside a loop.

### Google speech to text API


Here is the documentation for the python wrapper of the speech-to-text API.
https://cloud.google.com/speech-to-text/docs/libraries#client-libraries-install-python

In [ ]:
!pip install google-cloud-speech soundfile

In [ ]:
import io

import soundfile as sf
from google.api_core.client_options import ClientOptions

In [ ]:

# ADD VALUE HERE
# the API key of your google cloud project
google_api_key = 'INSERT_YOUR_API_KEY_HERE'
if not google_api_key:
    # without this check an empty key falls back to other credentials and raises a
    # confusing "default credentials were not found" error further down
    raise ValueError('Add your google cloud API key to google_api_key above')

client_options = ClientOptions(api_key=google_api_key)

# mp3 is only supported by the beta API, so we decode the mp3 into flac,
# a lossless format that both the main and the beta API can read
audio_data, sample_rate = sf.read('kennedy_speech.mp3')
audio_file = 'kennedy_speech.flac'
sf.write(audio_file, audio_data, sample_rate)

#### Main API

In [ ]:
from google.cloud import speech

# instantiate a client
client = speech.SpeechClient(client_options=client_options)
# read the bytes of the audio file
with io.open(audio_file, 'rb') as f:
    audio_content = f.read()
# create an audio object containing the audio content
audio = speech.RecognitionAudio(content=audio_content)

# set the parameters for the request
config = speech.RecognitionConfig(
    encoding=speech.RecognitionConfig.AudioEncoding.ENCODING_UNSPECIFIED,
    sample_rate_hertz=44100,
    language_code='en-US',
    enable_automatic_punctuation=True,
    audio_channel_count=2,
    model='latest_long'
)

# send an API request
response = client.recognize(config=config, audio=audio)

In [ ]:
response

#### Beta API (some additional functionalities)

In [ ]:
from google.cloud import speech_v1p1beta1

# instantiate a client
client = speech_v1p1beta1.SpeechClient(client_options=client_options)
# read the bytes of the audio file
with io.open(audio_file, 'rb') as f:
    audio_content = f.read()
# create an audio object containing the audio content
audio = speech_v1p1beta1.RecognitionAudio(content=audio_content)
# set the parameters for the request
config = speech_v1p1beta1.RecognitionConfig(
    encoding=speech_v1p1beta1.RecognitionConfig.AudioEncoding.ENCODING_UNSPECIFIED,
    sample_rate_hertz=44100,
    language_code='en-US',
    enable_automatic_punctuation=True,
    enable_word_confidence=True,
    audio_channel_count=2,
    model='latest_long'
)

# send an API request
response = client.recognize(config=config, audio=audio)

In [ ]:
response